In [5]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from analytics.forecasting.base import SimpleForecaster
from _pool_common import (
    load_pool_data,
    build_pooled_train_stack,
    backtest_21d_rolling,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_BASELINE,
    ARTIFACTS_DIR,
    TICKERS,
)
SPAN = 20

In [6]:
# Load pool: all tickers stacked into one DataFrame
stacked = load_pool_data()
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL     1256
AMZN     1256
GOOGL    1256
JNJ      1256
JPM      1256
MSFT     1256
NVDA     1256
SPY      1256
WMT      1256
XOM      1256
dtype: int64


,timestamp,symbol,close
0,2021-03-09,AAPL,121.089996
1,2021-03-10,AAPL,119.980003
2,2021-03-11,AAPL,121.959999
3,2021-03-12,AAPL,121.029999
4,2021-03-15,AAPL,123.989998
5,2021-03-16,AAPL,125.570000
6,2021-03-17,AAPL,124.760002
7,2021-03-18,AAPL,120.529999
8,2021-03-19,AAPL,119.989998
9,2021-03-22,AAPL,123.389999


In [7]:
# Train once on pooled data: average close by date across assets (before 60-day test), fit one EWM
pooled_train = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_BASELINE)
if not pooled_train.empty:
    avg_series = pooled_train.groupby("timestamp")["close"].mean().sort_index()
    if len(avg_series) >= MIN_TRAIN_BASELINE:
        global_baseline_model = SimpleForecaster(span=SPAN, confidence_level=0.95)
        global_baseline_model.fit(avg_series)
        global_baseline_last = float(avg_series.iloc[-1])
    else:
        global_baseline_model = None
        global_baseline_last = None
else:
    global_baseline_model = None
    global_baseline_last = None


def get_forecast_baseline(context: pd.Series, horizon: int):
    if global_baseline_model is None or global_baseline_last is None:
        return []
    fc = global_baseline_model.forecast(periods=horizon)
    point = (fc.get("point_forecast") or []) if fc else []
    if not point:
        return []
    scale = float(context.iloc[-1]) / global_baseline_last
    return [float(x) * scale for x in point]


# Backtest: 60-day test window, rolling by 1 week; predict only (global model trained above)
model_name = "baseline"
all_preds = []
for sym in TICKERS:
    grp = stacked[stacked["symbol"] == sym]
    if grp.empty:
        continue
    close_ser = grp.set_index("timestamp")["close"]
    if isinstance(close_ser, pd.DataFrame):
        close_ser = close_ser.iloc[:, 0] if close_ser.shape[1] == 1 else close_ser[sym] if sym in close_ser.columns else close_ser.iloc[:, 0]
    prices = close_ser.astype(float).dropna().sort_index()
    if len(prices) < TEST_SIZE + MIN_TRAIN_BASELINE:
        continue
    pred = backtest_21d_rolling(
        prices, TEST_SIZE, FORECAST_HORIZON, ROLLING_STEP,
        MIN_TRAIN_BASELINE,
        get_forecast_fn=get_forecast_baseline,
    )
    if pred.empty:
        continue
    pred["symbol"] = sym
    all_preds.append(pred)

pred_baseline = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"])
print(pred_baseline.groupby("symbol").size() if not pred_baseline.empty else "No predictions.")
pred_baseline.head()

symbol
AAPL     126
AMZN     126
GOOGL    126
JNJ      126
JPM      126
MSFT     126
NVDA     126
SPY      126
WMT      126
XOM      126
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-10,278.779999,275.637441,0,AAPL
1,2025-12-11,278.029999,275.637441,0,AAPL
2,2025-12-12,278.279999,275.637441,0,AAPL
3,2025-12-15,274.109985,275.637441,0,AAPL
4,2025-12-16,274.609985,275.637441,0,AAPL


In [8]:
# Metrics per symbol and overall: averaged over rolling mini-windows (MAE, RMSE, MAPE_%)
metrics_rows = []
for sym in pred_baseline["symbol"].unique():
    sub = pred_baseline[pred_baseline["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_baseline)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_baseline_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_baseline_pool.parquet")

       model   symbol        MAE       RMSE    MAPE_%
0   baseline     AAPL  10.614806  12.612144  4.019096
1   baseline     MSFT  21.562252  25.439666  5.062936
2   baseline    GOOGL  13.646728  15.443446  4.253209
3   baseline     AMZN  13.401750  15.751148  6.099282
4   baseline      JPM  13.725844  15.328880  4.346785
5   baseline      JNJ  13.146178  14.688752  5.727865
6   baseline      WMT   5.536552   6.305807  4.475501
7   baseline      SPY   7.681315   8.611282  1.114003
8   baseline      XOM   8.967677  10.358099  6.434059
9   baseline     NVDA   5.956749   6.895157  3.228526
10  baseline  overall  11.423985  15.081298  4.476126
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_baseline_pool.parquet
